In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 23 — ANÁLISE DE ERROS: APRENDENDO COM AS FALHAS DO MODELO

> "Um bom cientista celebra seus sucessos. Um grande cientista estuda suas falhas. É nos erros que residem as pistas para a próxima descoberta."
> — Um velho professor de laboratório

Meu ensemble era o auge do trabalho, com o melhor desempenho que eu alcançara. Eu estava orgulhoso. Mas o orgulho é um luxo que um cientista de dados na saúde não pode se permitir. A perfeição é inatingível — e isso significava que meu modelo ainda errava. Cada erro era um paciente virtual com diagnóstico errado.

Até aqui, eu tratara os erros como números numa matriz. Mas quem eram eles? Que características tinham os tumores que meu comitê ainda não decifrava? Estavam na fronteira entre benigno e maligno? Eram *outliers*? Havia um padrão? Meu trabalho não terminava no melhor modelo: terminava no entendimento profundo de suas limitações.

## Análise de Erros: A Fronteira Final da Melhoria

Análise de erros é examinar as previsões incorretas para obter *insights*. É uma das etapas mais valiosas — e mais negligenciadas — de um projeto. O processo: identificar os erros (com `cross_val_predict`, sem *leakage*), separá-los por tipo (FN vs FP) e comparar suas características com as dos acertos, à procura de padrões.

Vamos aplicar isso ao nosso modelo final, o ensemble, focando nos **Falsos Negativos** — os erros mais críticos.

#### Código 23.1: Investigando Nossas Falhas


In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import cross_val_predict
from oncoclassify_utils import X_train, y_train

log = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(solver="liblinear", random_state=42))])
svm = Pipeline([("sc", StandardScaler()), ("svm", SVC(kernel="rbf", C=10, gamma=0.01, probability=True, random_state=42))])
gb  = GradientBoostingClassifier(n_estimators=100, random_state=42)
comite = VotingClassifier([("lr", log), ("svm", svm), ("gb", gb)], voting="soft")

y_pred = cross_val_predict(comite, X_train, y_train, cv=5)
df = X_train.copy(); df["real"] = y_train.values; df["previsto"] = y_pred

erros = df[df.real != df.previsto]
fn = df[(df.real == 0) & (df.previsto == 1)]   # maligno previsto benigno
vp = df[(df.real == 0) & (df.previsto == 0)]   # maligno acertado
vn = df[(df.real == 1) & (df.previsto == 1)]   # benigno acertado
print(f"Erros totais: {len(erros)} | Falsos Negativos: {len(fn)} | Falsos Positivos: {len(df[(df.real==1)&(df.previsto==0)])}")

feats = ["worst concave points", "worst radius", "mean concave points"]
comp = pd.DataFrame({"FN (erro)": fn[feats].mean(),
                     "VP (maligno certo)": vp[feats].mean(),
                     "VN (benigno certo)": vn[feats].mean()})
print("\nMedia das features:")
print(comp.round(4).to_string())

Erros totais: 9 | Falsos Negativos: 6 | Falsos Positivos: 3

Media das features:
                      FN (erro)  VP (maligno certo)  VN (benigno certo)
worst concave points     0.0880              0.1866              0.0740
worst radius            15.5117             21.1656             13.3437
mean concave points      0.0321              0.0901              0.0256


> **⚠️ ATENÇÃO — Os erros moram na zona cinzenta**
>
> O padrão é revelador. Para cada *feature*, os **Falsos Negativos** têm valores **intermediários** entre os malignos que o modelo acerta e os benignos que ele acerta. Veja `worst concave points`: malignos acertados têm média **0,187**, benignos acertados **0,074**, e os malignos *errados*, **0,088** — bem no meio. Eles não são "obviamente" malignos nem benignos: vivem na fronteira da decisão. São tumores malignos com aparência atipicamente branda — exatamente os casos mais difíceis, como o de Helena.

**Hipótese acionável:** o modelo é fraco na região de fronteira. Para melhorar, valeria coletar mais casos ambíguos desse tipo, ou investigar *features* adicionais (talvez de imagem ou clínicas) capazes de separar essa zona cinzenta — não mais algoritmos, mas mais informação onde ela falta.

> **📌 CHECKLIST DO CAPÍTULO 23**
>
> ✅ Entende o propósito da análise de erros: ir além das métricas para entender *por que* o modelo falha.
>
> ✅ Sabe identificar e segmentar as previsões incorretas com `cross_val_predict`.
>
> ✅ Executou o Código 23.1 e identificou os **6 Falsos Negativos** do modelo final.
>
> ✅ Formulou uma hipótese sobre a causa (casos ambíguos na fronteira).
>
> **⚠️ CRÍTICO:** A análise de erros transforma o *machine learning* num trabalho de detetive. É um ciclo: o modelo erra, você analisa, gera hipóteses, melhora — e recomeça.

Olhar os dados dos Falsos Negativos foi uma lição de humildade. Não eram só números: eram os fantasmas dos erros do modelo. Mas, ao estudá-los, dei-lhes um propósito. Eles não eram mais falhas — eram professores, mostrando exatamente onde meu modelo, com toda a sua otimização, ainda era vulnerável.

Com isso, meu modelo estava completo: escolhido, otimizado, calibrado, explicado — e agora com suas fraquezas mapeadas. Mas havia algo que eu vinha adiando desde o Capítulo 4. Em todos esses meses, cada decisão — cada gráfico, cada *feature*, cada hiperparâmetro, cada limiar — fora tomada olhando **apenas** o conjunto de treino. Eu nunca havia visto meu modelo diante de dados verdadeiramente novos. O cofre continuava lacrado.

Chegara a hora de abri-lo. Uma única vez, sem volta. O julgamento final.
